In [5]:
import pandas as pd
import numpy as np
import os
import json

# Define our target round and target jurisdictions for the comparative coding
GD_NUM = "3" 
TARGET_COUNTRIES = ['US', 'GB', 'BR', 'CN', 'EU', 'IN'] # Adjust to match dataset country codes

data_dir = f"Data/GD{GD_NUM}"
print(f"Targeting data directory: {data_dir}")

# Verify file presence
expected_files = [
    f"GD{GD_NUM}_aggregate_standardized.csv",
    f"GD{GD_NUM}_participants.csv",
    f"GD{GD_NUM}_verbatim_map.csv"
]

for file in expected_files:
    path = os.path.join(data_dir, file)
    print(f"  - {file}: {'✅ Found' if os.path.exists(path) else '❌ Missing'}")

Targeting data directory: Data/GD3
  - GD3_aggregate_standardized.csv: ✅ Found
  - GD3_participants.csv: ✅ Found
  - GD3_verbatim_map.csv: ✅ Found


In [9]:
import pandas as pd
import os

# Load data (handling the low_memory warning)
df_agg = pd.read_csv(
    os.path.join(data_dir, f"GD{GD_NUM}_aggregate_standardized.csv"), 
    low_memory=False
)

# Core keywords looking for governance, institutional intermediaries, and trust
keywords = ['trust', 'responsible', 'government', 'employer', 'company', 'regulate', 'authority', 'institution', 'rules', 'law']

# Filter rows where either the Question OR the Response text mentions our keywords
matches = df_agg[
    df_agg['Question'].str.contains('|'.join(keywords), case=False, na=False) |
    df_agg['Response'].str.contains('|'.join(keywords), case=False, na=False)
]

print(f"Found {matches['Question ID'].nunique()} unique questions matching your framework.\n")
print("--- Sample of Relevant Questions & Responses Found ---")
print("-" * 80)

# Group by Question to see the distinct prompt texts
unique_questions = matches[['Question ID', 'Question']].drop_duplicates().head(10)

for idx, row in unique_questions.iterrows():
    q_id = row['Question ID']
    q_text = row['Question']
    print(f"[ID: {q_id}] {q_text}")
    
    # Grab a couple sample responses for this specific question
    sample_responses = matches[matches['Question ID'] == q_id]['Response'].unique()[:2]
    for resp in sample_responses:
        print(f"   ↳ Response item: \"{str(resp)[:100]}...\"")
    print("-" * 80)

Found 32 unique questions matching your framework.

--- Sample of Relevant Questions & Responses Found ---
--------------------------------------------------------------------------------
[ID: 074a34ab-92f8-4c30-bac0-59c9850802cb] What country or region do you most identify with?
   ↳ Response item: "Malawi..."
--------------------------------------------------------------------------------
[ID: 91fcb570-ec08-458e-af99-d9df0650384a] What has been the most noticeable change in your daily life, if any, as a result of AI in the past year?
   ↳ Response item: "the most noticeable thing about the rampant plague of AI has been how distrusting I've become of ima..."
   ↳ Response item: "One of the most obvious one has been the transformation of the content i consume on my social medias..."
--------------------------------------------------------------------------------
[ID: b1869877-1015-43ec-9645-84bf2db39c6f] To what extent, if at all, do you generally trust governments to do what is right?

In [12]:
import pandas as pd
import numpy as np
import os

# Load participant data
df_part = pd.read_csv(os.path.join(data_dir, f"GD{GD_NUM}_participants.csv"), low_memory=False)

# Define exact column mappings discovered in diagnostics
country_col = 'What country or region do you most identify with?'
gov_trust_col = 'To what extent, if at all, do you generally trust governments to do what is right?'
corp_trust_col = 'To what extent, if at all, do you generally trust large corporations to do what is right?'

# Target jurisdictions for comparative work
target_jurisdictions = ['India', 'China', 'United States', 'Brazil', 'United Kingdom']
df_filtered = df_part[df_part[country_col].isin(target_jurisdictions)].copy()

print(f"📊 Analyzing trust architectures across {len(df_filtered)} target jurisdiction respondents.\n")

# Map trust responses to a numeric scale if they are strings (e.g., "A great deal" -> 4, "Not at all" -> 1)
# Let's print unique values first to verify the scale
print("Trust column sample categories:")
print(df_filtered[gov_trust_col].unique())
print("-" * 60)

# Distribution breakdown
country_distribution = df_filtered[country_col].value_counts()
display(country_distribution.to_frame(name='Respondent Count'))

📊 Analyzing trust architectures across 360 target jurisdiction respondents.

Trust column sample categories:
<StringArray>
[         'Strongly Distrust',             'Somewhat Trust',
                         '--',          'Somewhat Distrust',
 'Neither Trust Nor Distrust',             'Strongly Trust']
Length: 6, dtype: str
------------------------------------------------------------


,Respondent Count
What country or region do you most identify with?,
India,183
China,70
United States,52
Brazil,30
United Kingdom,25


In [13]:
# Lexicon tracking mentions of downstream institutional intermediaries
intermediary_lexicon = {
    'Workplace / Employers': ['employer', 'boss', 'company', 'job', 'workplace', 'hr', 'office', 'manager'],
    'Education / Schools': ['school', 'university', 'college', 'teacher', 'professor', 'student', 'class'],
    'Healthcare': ['hospital', 'doctor', 'clinic', 'medical', 'nurse', 'patient', 'healthcare']
}

verbatim_col = 'What has been the most noticeable change in your daily life, if any, as a result of AI in the past year? (English)'

print("🔍 Scanning open-ended responses for organic mentions of institutional intermediaries...\n")

for category, keywords in intermediary_lexicon.items():
    pattern = '|'.join(keywords)
    # Filter for rows matching our intermediary keywords
    matches = df_filtered[df_filtered[verbatim_col].str.contains(pattern, case=False, na=False)]
    
    match_count = len(matches)
    pct = (match_count / len(df_filtered)) * 100
    print(f"📂 {category}: {match_count} mentions ({pct:.1f}% of target respondents)")
    
    if match_count > 0:
        # Display 2 clean examples
        samples = matches[[country_col, verbatim_col]].sample(min(2, match_count))
        for _, row in samples.iterrows():
            print(f"   ↳ [{row[country_col]}]: \"{row[verbatim_col].strip()}\"")
    print("-" * 80)

🔍 Scanning open-ended responses for organic mentions of institutional intermediaries...

📂 Workplace / Employers: 22 mentions (6.1% of target respondents)
   ↳ [India]: "It helps me consolidate my ideas in form of points. AI also helps me in understanding some complex ideas and topics in simple manner but what I've personally observed is AI tends to give incorrect answers at times, so fact checking personally is mandated.

AI has really made it easy to search information. Compared to traditional search engines, where we have to go through articles to look what we're looking for. AI gives direct and straight to the point answers."
   ↳ [China]: "Help me with my job"
--------------------------------------------------------------------------------
📂 Education / Schools: 11 mentions (3.1% of target respondents)
   ↳ [India]: "Most of the students are using AI to learn anything they wanted to by using AI"
   ↳ [India]: "Less intellectual growth in myself. I don't feel the need to know anymo